<a href="https://colab.research.google.com/github/menna890/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
One row = one content item (content_hash_id) over March 2026.

The warehouse daily table has grain report_date × client_hash_id × content_hash_id.
We restrict to rows where gsc_data_available IS TRUE so impressions, clicks, and
position reflect measured Search Console data—not zero-filled gaps before tracking
started. We then aggregate that month to one page-level row (sum of impressions and
clicks, mean position) because the decision is page-level: which page to fix first.

Pages with no reliable search measurement are excluded on purpose; more unfiltered
rows would add noise, not signal, for a search-performance gap analysis.

In [1]:
import duckdb
import google.colab.userdata

con = duckdb.connect()

hf_token = google.colab.userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"


con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      9841378 │
└──────────────┘



In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()
hf_token = google.colab.userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"


df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
""").df()

print(df.shape)
print(df.head())
print(df.columns.tolist())



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 31)
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0              

In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_data_available,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
WHERE gsc_data_available IS TRUE
""").df()

print(df.shape)
print(df.head())
print(df[["gsc_impressions", "gsc_clicks", "gsc_avg_position"]].describe())

(3611061, 7)
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   gsc_data_available  gsc_impressions  gsc_clicks  gsc_avg_position  
0                True               20           0          3.350000  
1                True                1           0          0.000000  
2                True              125           1          4.928000  
3                True                7           0          4.000000  
4                True               11           0          2.272727  
       gsc_impressions    gsc_clicks  gsc_avg_position
count     3.611061e+06  3.611061e+06      3.611061e+06
mean      7.772164e+01  2.275874e-01      

### Unit of analysis:
One row = one content item (content_hash_id) over March 2026.

The warehouse daily table has grain report_date × client_hash_id × content_hash_id.
We aggregate that month to one row per content item (sum impressions/clicks, mean position)
because the decision is page-level: which page to fix first.

In [4]:
pages = (
    df.groupby(["client_hash_id", "content_hash_id"], as_index=False)
      .agg(
          impressions=("gsc_impressions", "sum"),
          clicks=("gsc_clicks", "sum"),
          avg_position=("gsc_avg_position", "mean"),
          days=("report_date", "nunique"),
      )
)

pages["ctr"] = pages.apply(
    lambda r: (100.0 * r["clicks"] / r["impressions"]) if r["impressions"] > 0 else None,
    axis=1,
)

print(pages.shape)
pages.head()

(176738, 7)


,client_hash_id,content_hash_id,impressions,clicks,avg_position,days,ctr
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,1,0.00000
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,31,0.60423
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,6,0.00000
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,30,0.00000
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,31,0.00000


## 2. Fields: feature / label / context / excluded

Features: gsc_avg_position (knowable search rank), gsc_clicks, ga4_sessions (past performance metrics), word_count, and content_type (knowable at creation). Safe to use.

Label / Proxy: The observed performance gap (e.g., future CTR drop or position decline).

Context: client_hash_id, content_hash_id, report_date. Used for grouping, joining, and reading only — never for the model to learn from.

Excluded: trend_pct and trend_direction (because they are used to compute the label; using them causes leakage). Any product-decision flags (like health_score) are also excluded because they represent the human rules we are trying to beat, not the actual world.

In [5]:
# 1. Building the 5 features + the leakage trap
features_query = f"""
SELECT
    -- Context
    client_hash_id, content_hash_id, report_date,

    -- Features (Safe and knowable)
    gsc_avg_position,
    gsc_clicks,
    gsc_impressions,
    ga4_sessions,

    -- THE TRAP: Adding a target-derived column intentionally (Leakage)
    -- Simulating a column that looks into the future or defines the label
    (gsc_clicks / NULLIF(gsc_impressions, 0)) AS current_ctr_trap
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
WHERE ga4_data_available IS TRUE
LIMIT 10
"""

df_features = con.sql(features_query).df()
print("Columns WITH the leakage trap:", df_features.columns.tolist())

# 2. Removing the trap to keep the honest numbers
df_features = df_features.drop(columns=['current_ctr_trap'])
print("Honest columns AFTER removing the trap:", df_features.columns.tolist())

Columns WITH the leakage trap: ['client_hash_id', 'content_hash_id', 'report_date', 'gsc_avg_position', 'gsc_clicks', 'gsc_impressions', 'ga4_sessions', 'current_ctr_trap']
Honest columns AFTER removing the trap: ['client_hash_id', 'content_hash_id', 'report_date', 'gsc_avg_position', 'gsc_clicks', 'gsc_impressions', 'ga4_sessions']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# Verifying Windows, Counts, Missingness, and Grain in one powerful query
verify_contract_query = f"""
SELECT
    -- 1. Windows: Checking the date boundaries
    MIN(report_date) AS window_start,
    MAX(report_date) AS window_end,

    -- 2. Counts: Total rows and unique pages
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages,

    -- 3. Missing Values: Checking nulls in our main feature
    ROUND(AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS missing_position_pct,

    -- 4. Grain: Must be exactly 0 (proving no duplicates exist for client + content + date)
    (SELECT COUNT(*) FROM (
        SELECT client_hash_id, content_hash_id, report_date
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )) AS grain_violations
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')
WHERE ga4_data_available IS TRUE
"""

display(con.sql(verify_contract_query).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,window_start,window_end,total_rows,unique_pages,missing_position_pct,grain_violations
0,2026-03-01,2026-03-31,413966,90489,11.99,0


## 4. Data limits

1. Varying History Depth: Client onboarding dates vary significantly (gsc_data_start). A global 90-day lookback window will fail for new clients; we must use per-client windows.

2. GA4 Missingness Trap: Rows before a client's ga4_data_start are zero-filled with ga4_data_available = FALSE. These zeros represent "no tracking setup," not "zero traffic." Filtering with IS TRUE is mandatory.

3. Window Overlaps: When building labels (e.g., next 30 days performance), we must ensure our training window does not accidentally bleed into the sealed test month (June 2026).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.